# CTG-Net Architecture for 3-Class CTG Classification

This notebook evaluates the baseline CTG-Net architecture on the 3-class (Normal / Mild / Severe)
classification task, using the same preprocessing and cross-validation pipeline as the final model.

## 1. Imports

In [ ]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import numpy as np
import pandas as pd
from pathlib import Path
import wfdb
import json
import warnings
from typing import Dict, List, Tuple, Optional

from scipy.interpolate import interp1d

from sklearn.metrics import (
    confusion_matrix, f1_score, accuracy_score,
    balanced_accuracy_score, precision_recall_fscore_support,
)
from sklearn.utils.class_weight import compute_class_weight

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = True
    print("StratifiedGroupKFold available.")
except ImportError:
    from sklearn.model_selection import GroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = False
    print("WARNING: StratifiedGroupKFold not available, falling back to GroupKFold.")

import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    Conv1D, SeparableConv1D, BatchNormalization, Activation,
    AveragePooling1D, Dropout, GlobalAveragePooling1D, Dense,
    Reshape, Multiply,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, Callback,
)
from tensorflow.keras.utils import to_categorical

import matplotlib.pyplot as plt
import seaborn as sns

GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
tf.random.set_seed(GLOBAL_SEED)
warnings.filterwarnings('ignore')

print(f"TensorFlow {tf.__version__} | NumPy {np.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 2. Configuration

In [ ]:
RAW_DATASET  = Path("../data/raw_dataset")
STEP2_LABELS = Path("../ExpertAnnotations/step2_labels.csv")
STEP3_LABELS = Path("../ExpertAnnotations/step3_labels.csv")
OUTPUT_DIR   = Path("outputs/ctg_net_multiclass")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
RAW_SAMPLING_RATE    = 4
TARGET_SAMPLING_RATE = 1
DOWNSAMPLE_FACTOR    = RAW_SAMPLING_RATE // TARGET_SAMPLING_RATE
WINDOW_MINUTES       = 30
WINDOW_LENGTH        = WINDOW_MINUTES * 60 * TARGET_SAMPLING_RATE
SAMPLING_RATE        = TARGET_SAMPLING_RATE
SEGMENT_MINUTES      = WINDOW_MINUTES
SEGMENT_LENGTH       = WINDOW_LENGTH

In [ ]:
NUM_CLASSES = 3
CLASS_NAMES = ["Normal", "Mild", "Severe"]
LABEL_MAP   = {1: 0, 2: 1, 3: 2}

In [ ]:
CFG = dict(
    step_numbers            = [2, 3],
    mode_a_offsets          = {2: (60, 30), 3: (30, 0)},
    controlled_step_offsets = {
        2: [(70, 40), (60, 30), (50, 20)],
        3: [(50, 20), (40, 10), (30, 0)],
    },
    segment_minutes         = SEGMENT_MINUTES,
    segment_length          = SEGMENT_LENGTH,
    max_nan_fraction        = 0.16,
    pad_short_records       = False,
    n_splits                = 10,
    cv_seed                 = 42,
    min_severe_records      = 10,
    max_cv_attempts         = 100,
    ctg_net_dropout         = 0.25,
    use_clinical_features   = False,
    oversample_severe       = False,
    severe_boost            = 0.0,
    augment_train           = True,
    aug_time_warp           = True,
    aug_warp_sigma          = 0.10,
    aug_amplitude_scale     = True,
    aug_amp_sigma           = 0.10,
    aug_noise               = True,
    aug_noise_std           = 0.02,
    aug_signal_loss         = True,
    aug_gap_frac            = 0.05,
    aug_n_gaps              = 2,
    scratch_epochs          = 100,
    scratch_lr              = 1e-3,
    scratch_batch           = 16,
    scratch_patience        = 20,
    label_smoothing         = 0.05,
    use_class_weights       = True,
)

windows_per_step = {s: len(v) for s, v in CFG['controlled_step_offsets'].items()}
assert len(set(windows_per_step.values())) == 1, "All steps must use the same number of windows."
CONTROLLED_WINDOWS_PER_STEP = next(iter(windows_per_step.values()))

print("Configuration loaded.")
print(f"  Segment length: {CFG['segment_length']} samples ({CFG['segment_minutes']} min)")
print(f"  CV folds: {CFG['n_splits']} | Controlled windows/step: {CONTROLLED_WINDOWS_PER_STEP}")
print(f"  Model: CTG-Net (1D adapted, 3-class) | Clinical features: disabled")

## 3. Data Loading

In [ ]:
def load_step_labels(step_paths: Dict[int, Path]) -> Dict[int, pd.DataFrame]:
    step_labels = {}
    for step_num, path in step_paths.items():
        df = pd.read_csv(path)
        total = len(df)
        df = df[df['Majority_Vote_Label'] != -1].copy()
        df['class_id'] = df['Majority_Vote_Label'].map(LABEL_MAP)
        df['rec_id'] = df['rec_id'].astype(str)
        excluded = total - len(df)
        print(f"Step {step_num}: {len(df)} labelled records ({excluded} uninterpretable excluded)")
        for cls_id, name in enumerate(CLASS_NAMES):
            n = int((df['class_id'] == cls_id).sum())
            print(f"  Class {cls_id} ({name}): {n}")
        step_labels[step_num] = df
    return step_labels

In [ ]:
def remove_trailing_zeros(signal) -> list:
    sig = list(signal) if isinstance(signal, np.ndarray) else list(signal)
    i = len(sig) - 1
    while i >= 0 and sig[i] == 0:
        i -= 1
    return sig[:i + 1]

In [ ]:
def clean_fhr(fhr_array, fs: int = 4) -> np.ndarray:
    fhr = pd.Series(np.array(fhr_array, dtype=float))
    fhr.replace(0, np.nan, inplace=True)
    na = fhr.isnull()
    gap_groups = na.ne(na.shift()).cumsum()
    gap_sizes = fhr.groupby(gap_groups.values).transform('size')
    fhr = fhr[~(gap_sizes.ge(fs * 15 + 1) & na)].reset_index(drop=True)
    fhr[fhr < 50] = np.nan
    fhr[fhr > 200] = np.nan
    fhr = fhr.interpolate(method='linear')
    diff = fhr - fhr.shift()
    fhr[(diff > 25) | (diff < -25)] = np.nan
    fhr = fhr.interpolate(method='linear')
    fhr = fhr.ffill().bfill()
    return fhr.values

In [ ]:
def downsample_signal(signal: np.ndarray, factor: int = 4) -> np.ndarray:
    if factor <= 1:
        return np.asarray(signal, dtype=np.float32)
    return np.asarray(signal[::factor], dtype=np.float32)

In [ ]:
def load_raw_signals(dataset_path: Path) -> Dict[str, dict]:
    records = [p.stem for p in dataset_path.glob("*.hea")]
    print(f"Found {len(records)} .hea files in {dataset_path}")
    data = {}
    for rid in sorted(records):
        try:
            rec = wfdb.rdrecord(str(dataset_path / rid))
            fhr_raw = remove_trailing_zeros(rec.p_signal[:, 0].tolist())
            fhr_clean = clean_fhr(fhr_raw, fs=RAW_SAMPLING_RATE)
            fhr_ds = downsample_signal(fhr_clean, factor=DOWNSAMPLE_FACTOR)
            data[rid] = {'FHR': fhr_ds, 'length': len(fhr_ds)}
        except Exception as e:
            print(f"  Skipping {rid}: {e}")
    print(f"Loaded {len(data)} FHR signals")
    lengths = [d['length'] for d in data.values()]
    print(f"  Duration range: {min(lengths)/(SAMPLING_RATE*60):.1f} - {max(lengths)/(SAMPLING_RATE*60):.1f} min")
    return data

In [ ]:
STEP_LABEL_PATHS = {2: STEP2_LABELS, 3: STEP3_LABELS}
step_label_dfs = load_step_labels(STEP_LABEL_PATHS)
signal_data = load_raw_signals(RAW_DATASET)

labelled_rec_ids = set()
for df in step_label_dfs.values():
    labelled_rec_ids.update(df['rec_id'].tolist())

valid_rec_ids = sorted(labelled_rec_ids & set(signal_data.keys()))
print(f"Records with Step 2/3 label and signal: {len(valid_rec_ids)}")

## 4. Window Extraction

In [ ]:
def _window_bounds_from_end(
    sig_len: int,
    offsets_from_end_min: Tuple[int, int],
    fs: int = 4,
) -> Optional[Tuple[int, int]]:
    start_from_end_min, end_from_end_min = offsets_from_end_min
    if start_from_end_min <= end_from_end_min:
        raise ValueError(f"Expected start offset > end offset, got {offsets_from_end_min}")
    start_from_end = start_from_end_min * 60 * fs
    end_from_end = end_from_end_min * 60 * fs
    if sig_len < start_from_end:
        return None
    start_idx = sig_len - start_from_end
    end_idx = sig_len - end_from_end if end_from_end > 0 else sig_len
    if end_idx <= start_idx:
        return None
    return int(start_idx), int(end_idx)

In [ ]:
def _get_controlled_window_bounds(
    sig_len: int,
    step_num: int,
    controlled_step_offsets: Dict[int, List[Tuple[int, int]]],
    fs: int = 4,
) -> Optional[List[Tuple]]:
    if step_num not in controlled_step_offsets:
        raise KeyError(f"Missing controlled offsets for step {step_num}")
    bounds = []
    for window_index, offsets in enumerate(controlled_step_offsets[step_num]):
        cur_bounds = _window_bounds_from_end(sig_len, offsets, fs=fs)
        if cur_bounds is None:
            return None
        start_idx, end_idx = cur_bounds
        bounds.append((window_index, offsets, start_idx, end_idx))
    return bounds

In [ ]:
def create_step_windows(
    signal_data: Dict[str, dict],
    step_label_dfs: Dict[int, pd.DataFrame],
    record_ids: List[str],
    window_length: int = 7200,
    step_numbers: Optional[List[int]] = None,
    step_offsets: Optional[Dict[int, Tuple[int, int]]] = None,
    controlled_step_offsets: Optional[Dict[int, List[Tuple[int, int]]]] = None,
    max_nan_fraction: float = 0.16,
    pad_short_records: bool = False,
    fs: int = 4,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    if step_numbers is None:
        step_numbers = [2, 3]
    if step_offsets is None:
        step_offsets = {2: (60, 30), 3: (30, 0)}
    if controlled_step_offsets is None:
        controlled_step_offsets = {s: [step_offsets[s]] for s in step_numbers}

    windows_per_step = {s: len(controlled_step_offsets[s]) for s in step_numbers}
    assert len(set(windows_per_step.values())) == 1
    fixed_windows_per_step = next(iter(windows_per_step.values()))

    X_list, y_list, group_list, meta_list = [], [], [], []
    accepted = 0
    skipped_short = 0
    skipped_nan = 0

    for rid in record_ids:
        if rid not in signal_data:
            continue
        fhr = np.array(signal_data[rid]['FHR'], dtype=np.float64)
        sig_len = len(fhr)

        for step_num in step_numbers:
            step_df = step_label_dfs.get(step_num)
            if step_df is None:
                continue
            row = step_df[step_df['rec_id'] == rid]
            if row.empty:
                continue

            controlled_bounds = _get_controlled_window_bounds(
                sig_len, step_num, controlled_step_offsets, fs=fs
            )
            if controlled_bounds is None:
                skipped_short += 1
                continue

            label = int(row['class_id'].iloc[0])
            step_windows = []
            failed_short = False
            failed_nan = False

            for window_index, offsets, start_idx, end_idx in controlled_bounds:
                window = fhr[start_idx:end_idx]
                if len(window) < window_length:
                    if pad_short_records:
                        pad_len = window_length - len(window)
                        window = np.pad(window, (pad_len, 0), mode='edge')
                    else:
                        failed_short = True
                        break
                missing_frac = float(np.mean(np.isnan(window) | (window == 0)))
                if missing_frac > max_nan_fraction:
                    failed_nan = True
                    break
                if np.any(np.isnan(window)):
                    window = pd.Series(window).interpolate(method='linear').ffill().bfill().values
                step_windows.append({
                    'window': window.astype(np.float32),
                    'window_index': window_index,
                    'offset_start_min': int(offsets[0]),
                    'offset_end_min': int(offsets[1]),
                    'window_start': start_idx,
                    'window_end': end_idx,
                    'missing_frac_prefill': missing_frac,
                })

            if failed_short:
                skipped_short += 1
                continue
            if failed_nan:
                skipped_nan += 1
                continue

            accepted += 1
            for sw in step_windows:
                X_list.append(sw['window'])
                y_list.append(label)
                group_list.append(rid)
                meta_list.append({
                    'rec_id': rid, 'step': step_num,
                    'window_index': sw['window_index'],
                    'offset_start_min': sw['offset_start_min'],
                    'offset_end_min': sw['offset_end_min'],
                    'window_start': sw['window_start'],
                    'window_end': sw['window_end'],
                    'label': label,
                    'missing_frac_prefill': sw['missing_frac_prefill'],
                })

    X_windows = np.array(X_list, dtype=np.float32)[:, :, np.newaxis] if X_list else np.empty((0, window_length, 1), dtype=np.float32)
    y_windows = np.array(y_list, dtype=np.int32)
    groups = np.array(group_list)
    metadata = pd.DataFrame(meta_list)

    print(f"Window extraction complete: {len(X_windows)} windows from {len(set(group_list))} records")
    print(f"  Accepted record-steps: {accepted} | Skipped short: {skipped_short} | Skipped NaN: {skipped_nan}")
    for cls_id, name in enumerate(CLASS_NAMES):
        print(f"  Class {cls_id} ({name}): {int(np.sum(y_windows == cls_id))} windows")
    return X_windows, y_windows, groups, metadata

In [ ]:
print("Extracting Step 2 + Step 3 controlled 30-minute windows...")
X_all, y_all, groups_all, meta_all = create_step_windows(
    signal_data=signal_data,
    step_label_dfs=step_label_dfs,
    record_ids=valid_rec_ids,
    window_length=CFG['segment_length'],
    step_numbers=CFG['step_numbers'],
    step_offsets=CFG['mode_a_offsets'],
    controlled_step_offsets=CFG['controlled_step_offsets'],
    max_nan_fraction=CFG['max_nan_fraction'],
    pad_short_records=CFG['pad_short_records'],
    fs=SAMPLING_RATE,
)
print(f"Window shape: {X_all.shape}")

## 5. Normalization and Augmentation

In [ ]:
def compute_norm_stats(X_train: np.ndarray) -> dict:
    vals = X_train.ravel()
    mean = float(np.mean(vals))
    std = float(np.std(vals))
    if std < 1e-8:
        std = 1.0
    return {'mean': mean, 'std': std}

In [ ]:
def apply_norm(X: np.ndarray, stats: dict) -> np.ndarray:
    X_norm = X.astype(np.float32).copy()
    X_norm -= stats['mean']
    X_norm /= stats['std']
    return X_norm

In [ ]:
def time_warp(X: np.ndarray, sigma: float = 0.1) -> np.ndarray:
    N, T, C = X.shape
    X_warped = np.empty_like(X)
    orig_steps = np.arange(T)
    n_knots = 4
    knot_pos = np.linspace(0, T - 1, n_knots + 2)
    for i in range(N):
        warp_factors = np.random.normal(loc=1.0, scale=sigma, size=n_knots + 2)
        warp_factors[0] = 1.0
        warp_factors[-1] = 1.0
        warped_knots = np.cumsum(np.diff(knot_pos) * warp_factors[:-1])
        warped_knots = np.concatenate([[0], warped_knots])
        warped_knots = warped_knots / warped_knots[-1] * (T - 1)
        interp_fn = interp1d(warped_knots, knot_pos, kind='linear', fill_value='extrapolate')
        warped_steps = np.clip(interp_fn(orig_steps), 0, T - 1)
        for c in range(C):
            interp_sig = interp1d(orig_steps, X[i, :, c], kind='linear', fill_value='extrapolate')
            X_warped[i, :, c] = interp_sig(warped_steps)
    return X_warped.astype(np.float32)

In [ ]:
def amplitude_scale(X: np.ndarray, sigma: float = 0.1) -> np.ndarray:
    scales = np.random.normal(1.0, sigma, size=(X.shape[0], 1, 1)).astype(np.float32)
    return X * scales

In [ ]:
def additive_noise(X: np.ndarray, noise_std: float = 0.02) -> np.ndarray:
    return X + np.random.normal(0, noise_std, X.shape).astype(np.float32)

In [ ]:
def signal_loss_simulation(X: np.ndarray, max_gap_frac: float = 0.05, n_gaps: int = 2) -> np.ndarray:
    N, T, C = X.shape
    X_aug = X.copy()
    max_gap = max(1, int(T * max_gap_frac))
    for i in range(N):
        for _ in range(n_gaps):
            gap_len = np.random.randint(1, max_gap + 1)
            start = np.random.randint(0, T - gap_len)
            end = start + gap_len
            for c in range(C):
                left_val = X_aug[i, max(start - 1, 0), c]
                right_val = X_aug[i, min(end, T - 1), c]
                X_aug[i, start:end, c] = np.linspace(left_val, right_val, gap_len)
    return X_aug.astype(np.float32)

In [ ]:
def augment_signal(X: np.ndarray, cfg: dict) -> np.ndarray:
    X_aug = X.copy()
    if cfg.get('aug_time_warp', True):
        X_aug = time_warp(X_aug, sigma=cfg.get('aug_warp_sigma', 0.1))
    if cfg.get('aug_amplitude_scale', True):
        X_aug = amplitude_scale(X_aug, sigma=cfg.get('aug_amp_sigma', 0.1))
    if cfg.get('aug_noise', True):
        X_aug = additive_noise(X_aug, noise_std=cfg.get('aug_noise_std', 0.02))
    if cfg.get('aug_signal_loss', True):
        X_aug = signal_loss_simulation(
            X_aug,
            max_gap_frac=cfg.get('aug_gap_frac', 0.05),
            n_gaps=cfg.get('aug_n_gaps', 2),
        )
    return X_aug

## 6. CTG-Net Architecture

1D adaptation of CTG-Net for FHR-only 3-class classification. The original CTG-Net uses 2D convolutions over FHR and UC. This version adapts the architecture to 1D for FHR-only input, preserving the temporal convolution, separable convolution, and pooling structure.

In [ ]:
def build_ctg_net_1d_multiclass(
    input_length: int = SEGMENT_LENGTH,
    num_classes: int = NUM_CLASSES,
    dropout_rate: float = 0.25,
) -> Model:
    """
    1D adaptation of CTG-Net for 3-class FHR classification.

    Adapted from the CTG-Net paper (Zhao et al.):
    - Conv2D(4, (1,3)) -> Conv1D(4, 3): temporal convolution
    - DepthwiseConv2D removed (FHR-UC cross-channel interaction is not applicable here)
    - SeparableConv2D(8, (1,3)) -> SeparableConv1D(8, 3)
    - AveragePooling2D -> AveragePooling1D
    - Flatten -> GlobalAveragePooling1D
    - Dense(2, sigmoid) -> Dense(3, softmax) for 3-class output
    """
    inputs = Input(shape=(input_length, 1), name="fhr_input")

    x = Conv1D(4, 3, padding="same", use_bias=False, name="temporal_conv")(inputs)
    x = BatchNormalization(name="temporal_bn")(x)
    x = Activation("relu", name="temporal_relu")(x)

    x = AveragePooling1D(pool_size=4, name="pool1")(x)
    x = Dropout(dropout_rate, name="dropout1")(x)

    x = SeparableConv1D(8, 3, padding="same", use_bias=False, name="sep_conv")(x)
    x = BatchNormalization(name="sep_bn")(x)
    x = Activation("relu", name="sep_relu")(x)

    x = AveragePooling1D(pool_size=4, name="pool2")(x)
    x = Dropout(dropout_rate, name="dropout2")(x)

    x = GlobalAveragePooling1D(name="global_pool")(x)
    outputs = Dense(num_classes, activation="softmax", name="output")(x)

    return Model(inputs, outputs, name="CTG-Net-1D-Multiclass")

In [ ]:
_test_model = build_ctg_net_1d_multiclass()
_test_model.summary()
print(f"Total parameters: {_test_model.count_params():,}")
del _test_model

## 7. Training Functions

In [ ]:
def _make_model_input(X, clin_feats=None):
    if clin_feats is not None:
        return [X, clin_feats]
    return X

In [ ]:
def _ensure_step_keys(metadata: pd.DataFrame) -> pd.DataFrame:
    meta = metadata.copy()
    if 'step_key' not in meta.columns:
        if 'rec_id' not in meta.columns or 'step' not in meta.columns:
            raise KeyError("metadata must contain 'rec_id' and 'step' columns")
        meta['step_key'] = meta['rec_id'].astype(str) + '__step' + meta['step'].astype(str)
    return meta

In [ ]:
def build_step_target_table(metadata: pd.DataFrame) -> pd.DataFrame:
    meta = _ensure_step_keys(metadata)
    if 'label' not in meta.columns:
        raise KeyError("metadata must contain a 'label' column")
    step_targets = meta[['step_key', 'rec_id', 'step', 'label']].drop_duplicates().copy()
    if step_targets['step_key'].duplicated().any():
        raise RuntimeError("Each step_key must map to exactly one label")
    return step_targets.sort_values(['rec_id', 'step']).reset_index(drop=True)

In [ ]:
def resolve_min_severe_threshold(
    metadata: pd.DataFrame,
    n_splits: int,
    requested_min: int = 10,
    severe_class: int = 2,
) -> Tuple[int, int, int]:
    step_targets = build_step_target_table(metadata.reset_index(drop=True))
    step_y = step_targets['label'].to_numpy(dtype=int)
    total_severe = int(np.sum(step_y == severe_class))
    min_possible = total_severe // n_splits if n_splits > 0 else 0
    effective_min = requested_min
    if effective_min > min_possible:
        effective_min = max(min_possible - 1, 1) if total_severe > 0 else 0
        print(f"  WARNING: Requested >= {requested_min} Severe step-units/fold is infeasible; using {effective_min}.")
    return effective_min, total_severe, min_possible

In [ ]:
def compute_class_weights_from_y(y_train: np.ndarray) -> dict:
    classes = np.unique(y_train)
    weights = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
    print(f"  Class weights (from train fold): {class_weight}")
    return class_weight

In [ ]:
class RecordStepBalancedAccuracy(Callback):
    def __init__(self, X_val, val_metadata, clin_val=None, batch_size=32):
        super().__init__()
        self.X_val = X_val
        self.val_metadata = _ensure_step_keys(val_metadata.reset_index(drop=True))
        self.clin_val = clin_val
        self.batch_size = batch_size

    def _aggregate(self, y_proba):
        step_y_true, step_y_pred = [], []
        for step_key in self.val_metadata['step_key'].unique():
            mask = self.val_metadata['step_key'] == step_key
            mean_proba = y_proba[mask].mean(axis=0)
            step_y_true.append(int(self.val_metadata.loc[mask, 'label'].iloc[0]))
            step_y_pred.append(int(np.argmax(mean_proba)))
        return np.array(step_y_true, dtype=np.int32), np.array(step_y_pred, dtype=np.int32)

    def on_epoch_end(self, epoch, logs=None):
        del epoch
        logs = logs or {}
        val_input = _make_model_input(self.X_val, self.clin_val)
        y_proba = self.model.predict(val_input, batch_size=self.batch_size, verbose=0)
        step_y_true, step_y_pred = self._aggregate(y_proba)
        bal_acc = float(balanced_accuracy_score(step_y_true, step_y_pred))
        logs['val_record_step_balanced_accuracy'] = bal_acc
        print(f"  val_record_step_balanced_accuracy: {bal_acc:.4f}")

In [ ]:
def get_callbacks(
    patience: int = 20,
    monitor: str = 'val_loss',
    monitor_mode: str = 'min',
    save_path=None,
    prefix_callbacks=None,
    save_weights_only: bool = False,
) -> list:
    callbacks = list(prefix_callbacks or [])
    callbacks.extend([
        EarlyStopping(
            monitor=monitor, mode=monitor_mode,
            patience=patience, restore_best_weights=True, verbose=1,
        ),
        ReduceLROnPlateau(
            monitor=monitor, mode=monitor_mode,
            factor=0.5, patience=max(patience // 2, 3),
            min_lr=1e-6, verbose=1,
        ),
    ])
    if save_path is not None:
        callbacks.append(ModelCheckpoint(
            save_path, monitor=monitor, mode=monitor_mode,
            save_best_only=True, save_weights_only=save_weights_only, verbose=0,
        ))
    return callbacks

In [ ]:
def oversample_minority_records(
    train_idx: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    severe_class: int = 2,
) -> np.ndarray:
    del groups
    y_train = y[train_idx]
    class_counts = {c: int(np.sum(y_train == c)) for c in range(NUM_CLASSES)}
    max_count = max(class_counts.values())
    severe_count = class_counts.get(severe_class, 0)
    if severe_count == 0 or severe_count >= max_count:
        return train_idx
    n_copies = max(1, round(max_count / severe_count) - 1)
    severe_indices = train_idx[y_train == severe_class]
    extra_indices = np.tile(severe_indices, n_copies)
    return np.concatenate([train_idx, extra_indices])

In [ ]:
def calibrate_predictions(
    y_proba: np.ndarray,
    severe_class: int = 2,
    severe_boost: float = 0.0,
) -> np.ndarray:
    if severe_boost <= 0:
        return y_proba
    adjusted = y_proba.copy()
    adjusted[:, severe_class] += severe_boost
    return adjusted / adjusted.sum(axis=1, keepdims=True)

In [ ]:
def train_from_scratch(
    model: Model,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    val_metadata: pd.DataFrame,
    cfg: dict,
    save_path=None,
    clin_train=None,
    clin_val=None,
) -> tf.keras.callbacks.History:
    print("\n" + "=" * 50)
    print("TRAINING FROM SCRATCH")
    print("=" * 50)

    label_smoothing = float(cfg.get('label_smoothing', 0.0))
    model.compile(
        optimizer=Adam(learning_rate=cfg['scratch_lr']),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics=['accuracy'],
    )

    class_weight = None
    if cfg.get('use_class_weights', True):
        class_weight = compute_class_weights_from_y(y_train)

    y_train_cat = to_categorical(y_train, NUM_CLASSES)
    y_val_cat = to_categorical(y_val, NUM_CLASSES)

    X_train_use = X_train.copy()
    if cfg.get('augment_train', False):
        X_train_use = augment_signal(X_train_use, cfg)

    monitor_callback = RecordStepBalancedAccuracy(
        X_val=X_val, val_metadata=val_metadata,
        clin_val=clin_val, batch_size=32,
    )
    print(f"  Early stopping monitor: val_record_step_balanced_accuracy")
    print(f"  Label smoothing: {label_smoothing:.2f}")

    history = model.fit(
        _make_model_input(X_train_use, clin_train),
        y_train_cat,
        validation_data=(_make_model_input(X_val, clin_val), y_val_cat),
        epochs=cfg['scratch_epochs'],
        batch_size=cfg['scratch_batch'],
        class_weight=class_weight,
        callbacks=get_callbacks(
            patience=cfg['scratch_patience'],
            monitor='val_record_step_balanced_accuracy',
            monitor_mode='max',
            save_path=save_path,
            prefix_callbacks=[monitor_callback],
            save_weights_only=True,
        ),
        verbose=1,
    )
    print(f"  Training finished at epoch {len(history.history['loss'])}")
    return history

In [ ]:
def stratified_record_kfold_with_constraint(
    metadata: pd.DataFrame,
    n_splits: int = 5,
    min_severe_records: int = 10,
    max_attempts: int = 100,
    cv_seed: int = 42,
    severe_class: int = 2,
) -> Tuple[List[Tuple[np.ndarray, np.ndarray]], int]:
    meta = _ensure_step_keys(metadata.reset_index(drop=True))
    step_targets = build_step_target_table(meta)
    step_y = step_targets['label'].to_numpy(dtype=int)
    step_groups = step_targets['rec_id'].to_numpy()
    step_keys = step_targets['step_key'].to_numpy()

    effective_min, total_severe, min_possible = resolve_min_severe_threshold(
        meta, n_splits=n_splits, requested_min=min_severe_records, severe_class=severe_class,
    )
    print(f"  Total Severe step-units: {total_severe}, {n_splits} folds -> max ~{min_possible}/fold")

    best_folds = None
    best_min_severe = -1
    best_counts = None
    n_attempts = max_attempts if HAS_STRATIFIED_GROUP_KFOLD else 1

    for attempt in range(n_attempts):
        seed = cv_seed + attempt
        if HAS_STRATIFIED_GROUP_KFOLD:
            splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
            split_iter = splitter.split(step_keys, step_y, step_groups)
        else:
            splitter = GroupKFold(n_splits=n_splits)
            split_iter = splitter.split(step_keys, step_y, step_groups)

        candidate_folds = []
        fold_severe_counts = []
        all_ok = True

        for train_step_idx, val_step_idx in split_iter:
            train_rec_ids = set(step_targets.iloc[train_step_idx]['rec_id'])
            val_rec_ids = set(step_targets.iloc[val_step_idx]['rec_id'])
            n_severe = int(np.sum(step_y[val_step_idx] == severe_class))
            fold_severe_counts.append(n_severe)
            train_mask = meta['rec_id'].isin(train_rec_ids).to_numpy()
            val_mask = meta['rec_id'].isin(val_rec_ids).to_numpy()
            candidate_folds.append((np.where(train_mask)[0], np.where(val_mask)[0]))
            if n_severe < effective_min:
                all_ok = False

        cur_min = min(fold_severe_counts) if fold_severe_counts else -1
        if cur_min > best_min_severe:
            best_min_severe = cur_min
            best_folds = candidate_folds
            best_counts = list(fold_severe_counts)

        if all_ok:
            method = "StratifiedGroupKFold" if HAS_STRATIFIED_GROUP_KFOLD else "GroupKFold"
            print(f"  Valid split at attempt {attempt + 1} ({method}): Severe per fold: {fold_severe_counts}")
            return candidate_folds, effective_min

    print(f"  Could not meet >= {effective_min} Severe/fold. Best: min={best_min_severe}, per-fold={best_counts}")
    return best_folds, effective_min

In [ ]:
def analyze_validation_set(
    metadata: pd.DataFrame,
    val_idx: np.ndarray,
    min_severe_records: int = 10,
    severe_class: int = 2,
) -> Tuple[bool, dict]:
    val_meta = _ensure_step_keys(metadata.iloc[val_idx].reset_index(drop=True))
    step_targets = build_step_target_table(val_meta)
    step_counts = {
        cls_id: int(np.sum(step_targets['label'].to_numpy(dtype=int) == cls_id))
        for cls_id in range(NUM_CLASSES)
    }
    window_counts = {
        cls_id: int(np.sum(val_meta['label'].to_numpy(dtype=int) == cls_id))
        for cls_id in range(NUM_CLASSES)
    }
    n_severe = step_counts.get(severe_class, 0)
    stats = dict(
        n_val_records=int(val_meta['rec_id'].nunique()),
        n_val_step_units=len(step_targets),
        n_val_windows=len(val_meta),
        step_units_per_class=step_counts,
        windows_per_class=window_counts,
        n_severe_step_units=n_severe,
        is_reliable=(n_severe >= min_severe_records),
        min_severe_threshold=min_severe_records,
    )
    return stats['is_reliable'], stats

## 8. Evaluation and Metrics

In [ ]:
def aggregate_to_record_step_level(
    y_proba: np.ndarray,
    metadata: pd.DataFrame,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    meta = _ensure_step_keys(metadata.reset_index(drop=True))
    unique_step_keys = meta['step_key'].unique()
    step_y_true, step_y_pred = [], []
    for step_key in unique_step_keys:
        mask = meta['step_key'] == step_key
        mean_proba = y_proba[mask].mean(axis=0)
        step_y_pred.append(int(np.argmax(mean_proba)))
        step_y_true.append(int(meta.loc[mask, 'label'].iloc[0]))
    return unique_step_keys, np.array(step_y_true, dtype=int), np.array(step_y_pred, dtype=int)

In [ ]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    labels = list(range(NUM_CLASSES))
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', labels=labels, zero_division=0)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    metrics = {'accuracy': acc, 'balanced_accuracy': bal_acc, 'macro_f1': macro_f1, 'confusion_matrix': cm}
    for i, name in enumerate(CLASS_NAMES):
        metrics[f'precision_{name}'] = float(precision[i])
        metrics[f'recall_{name}'] = float(recall[i])
        metrics[f'f1_{name}'] = float(f1[i])
        metrics[f'support_{name}'] = int(support[i]) if support[i] is not None else 0
    return metrics

In [ ]:
def print_metrics(metrics: dict, prefix: str = "") -> None:
    print(f"\n{prefix}Record-Step Metrics:")
    print(f"  Accuracy:          {metrics['accuracy']:.4f}")
    print(f"  Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
    print(f"  Macro F1:          {metrics['macro_f1']:.4f}")
    print(f"  " + "-" * 45)
    for name in CLASS_NAMES:
        print(f"  {name:8s}  P={metrics[f'precision_{name}']:.3f}  "
              f"R={metrics[f'recall_{name}']:.3f}  "
              f"F1={metrics[f'f1_{name}']:.3f}  "
              f"N={metrics.get(f'support_{name}', '?')}")

## 9. Visualisation

In [ ]:
def plot_confusion_matrix(
    cm: np.ndarray,
    title: str = "Confusion Matrix",
    save_path=None,
) -> None:
    _, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        ax=ax, linewidths=0.5,
    )
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.set_title(title, fontsize=14)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_fold_summary(fold_results: List[dict], save_path=None) -> None:
    f1s = [r['macro_f1'] for r in fold_results]
    folds = list(range(1, len(f1s) + 1))
    _, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(folds, f1s, color='steelblue', edgecolor='black', alpha=0.8)
    ax.axhline(
        np.mean(f1s), color='red', linestyle='--',
        label=f"Mean = {np.mean(f1s):.3f} +/- {np.std(f1s):.3f}",
    )
    ax.set_xlabel('Fold', fontsize=12)
    ax.set_ylabel('Macro F1', fontsize=12)
    ax.set_title('Per-Fold Record-Step Macro F1', fontsize=14)
    ax.set_xticks(folds)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, f1s):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f'{val:.3f}', ha='center', fontsize=10)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

## 10. Cross-Validation Pipeline

Grouped 10-fold cross-validation by `rec_id`. Evaluation is at the record-step level using mean probability aggregation.

In [ ]:
def run_ctg_net_cv(
    X: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    metadata: pd.DataFrame,
    cfg: dict,
) -> List[dict]:
    """Grouped cross-validation pipeline using CTG-Net as the model."""
    if len(X) != len(y) or len(X) != len(groups) or len(X) != len(metadata):
        raise ValueError("X, y, groups, and metadata must all have the same length")

    metadata = _ensure_step_keys(metadata.reset_index(drop=True))
    n_splits = cfg['n_splits']
    severe_boost = cfg.get('severe_boost', 0.0)
    do_oversample = cfg.get('oversample_severe', False)
    requested_min_severe = cfg.get('min_severe_records', 10)

    folds, effective_min_severe = stratified_record_kfold_with_constraint(
        metadata,
        n_splits=n_splits,
        min_severe_records=requested_min_severe,
        max_attempts=cfg.get('max_cv_attempts', 100),
        cv_seed=cfg['cv_seed'],
    )

    fold_results = []
    all_step_y_true, all_step_y_pred = [], []
    reliable_step_y_true, reliable_step_y_pred = [], []

    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        print(f"\n{'=' * 60}")
        print(f"FOLD {fold_idx + 1} / {n_splits}")
        print(f"{'=' * 60}")

        is_reliable, val_stats = analyze_validation_set(
            metadata, val_idx, min_severe_records=effective_min_severe,
        )
        tag = "RELIABLE" if is_reliable else "UNRELIABLE"
        print(f"  Validation quality: {tag}")
        for cls_id, cls_name in enumerate(CLASS_NAMES):
            print(f"    {cls_name}: {val_stats['step_units_per_class'][cls_id]} step-units, "
                  f"{val_stats['windows_per_class'][cls_id]} windows")
        print(f"  Severe step-units in val: {val_stats['n_severe_step_units']} (threshold: {effective_min_severe})")

        if do_oversample:
            train_idx = oversample_minority_records(train_idx, y, groups, severe_class=2)

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        g_train, g_val = groups[train_idx], groups[val_idx]
        meta_val = metadata.iloc[val_idx].reset_index(drop=True)

        overlap = set(g_train) & set(g_val)
        assert len(overlap) == 0, f"DATA LEAKAGE: {len(overlap)} records in both sets"

        print(f"  Train: {len(X_train)} windows from {len(set(g_train))} records")
        print(f"  Val:   {len(X_val)} windows from {len(set(g_val))} records")
        for cls_id, cls_name in enumerate(CLASS_NAMES):
            print(f"    {cls_name}: train={int(np.sum(y_train == cls_id))}, val={int(np.sum(y_val == cls_id))}")

        norm_stats = compute_norm_stats(X_train)
        X_train_n = apply_norm(X_train, norm_stats)
        X_val_n = apply_norm(X_val, norm_stats)

        tf.keras.backend.clear_session()
        model = build_ctg_net_1d_multiclass(
            input_length=cfg['segment_length'],
            num_classes=NUM_CLASSES,
            dropout_rate=cfg.get('ctg_net_dropout', 0.25),
        )

        save_path = str(OUTPUT_DIR / f"fold{fold_idx + 1}_best.weights.h5")
        _ = train_from_scratch(
            model, X_train_n, y_train, X_val_n, y_val, meta_val, cfg, save_path=save_path,
        )

        y_proba_val = model.predict(X_val_n, batch_size=32, verbose=0)
        if severe_boost > 0:
            y_proba_val = calibrate_predictions(y_proba_val, severe_class=2, severe_boost=severe_boost)

        _, step_y_true, step_y_pred = aggregate_to_record_step_level(y_proba_val, meta_val)
        metrics = compute_metrics(step_y_true, step_y_pred)
        metrics['fold'] = fold_idx + 1
        metrics['n_train_records'] = len(set(g_train))
        metrics['n_val_records'] = len(set(g_val))
        metrics['n_val_step_units'] = val_stats['n_val_step_units']
        metrics['is_reliable'] = is_reliable
        metrics['n_severe_val_step_units'] = val_stats['n_severe_step_units']
        fold_results.append(metrics)

        all_step_y_true.extend(step_y_true)
        all_step_y_pred.extend(step_y_pred)
        if is_reliable:
            reliable_step_y_true.extend(step_y_true)
            reliable_step_y_pred.extend(step_y_pred)

        print_metrics(metrics, prefix=f"Fold {fold_idx + 1} ")
        if not is_reliable:
            print(f"  Fold flagged UNRELIABLE (only {val_stats['n_severe_step_units']} Severe step-units)")

        plot_confusion_matrix(
            metrics['confusion_matrix'],
            title=f"CTG-Net Fold {fold_idx + 1}" + (" [UNRELIABLE]" if not is_reliable else ""),
            save_path=str(OUTPUT_DIR / f"fold{fold_idx + 1}_cm.png"),
        )

    n_reliable = sum(1 for r in fold_results if r['is_reliable'])
    reliable_results = [r for r in fold_results if r['is_reliable']]

    print(f"\n{'=' * 60}")
    print(f"CROSS-VALIDATION SUMMARY (CTG-Net)")
    print(f"  Reliable folds: {n_reliable} / {n_splits}")
    print(f"{'=' * 60}")

    for key in ['accuracy', 'balanced_accuracy', 'macro_f1']:
        vals = [r[key] for r in fold_results]
        print(f"  {key:22s}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}")

    overall_cm = confusion_matrix(all_step_y_true, all_step_y_pred, labels=list(range(NUM_CLASSES)))
    plot_confusion_matrix(
        overall_cm,
        title="CTG-Net Overall Record-Step CM (All Folds)",
        save_path=str(OUTPUT_DIR / "overall_cm.png"),
    )
    plot_fold_summary(fold_results, save_path=str(OUTPUT_DIR / "fold_summary.png"))
    return fold_results

## 11. Run Experiment

Runs the CTG-Net 3-class evaluation with manual class weights to handle class imbalance.

In [ ]:
step_unit_table = build_step_target_table(meta_all)
print(f"Dataset: {len(X_all)} windows from {len(step_unit_table)} record-steps across {len(np.unique(groups_all))} records")
print(f"Window shape: {X_all.shape}")
print(f"Classes: {dict(zip(CLASS_NAMES, [int(np.sum(y_all == i)) for i in range(NUM_CLASSES)]))}")

manual_class_weights = {0: 0.9, 1: 1.2, 2: 2.5}
_original_compute_class_weights_from_y = compute_class_weights_from_y

In [ ]:
def _manual_class_weight_resolver_factory(manual_weights):
    def _resolver(y_train: np.ndarray) -> dict:
        if manual_weights is None:
            return _original_compute_class_weights_from_y(y_train)
        resolved = {
            int(cls_idx): float(manual_weights.get(cls_idx, 1.0))
            for cls_idx in range(NUM_CLASSES)
        }
        print(f"  Class weights (manual override): {resolved}")
        return resolved
    return _resolver

In [ ]:
try:
    print("=" * 70)
    print("Running CTG-Net 3-class experiment")
    print(f"Manual class weights: {manual_class_weights}")
    print(f"Dropout rate: {CFG['ctg_net_dropout']}")
    print("=" * 70)

    compute_class_weights_from_y = _manual_class_weight_resolver_factory(manual_class_weights)
    ctg_net_results = run_ctg_net_cv(
        X=X_all, y=y_all, groups=groups_all, metadata=meta_all, cfg=CFG,
    )
finally:
    compute_class_weights_from_y = _original_compute_class_weights_from_y

reliable_results = [r for r in ctg_net_results if r.get('is_reliable', True)]
ctg_net_summary = {
    'experiment': 'ctg_net_multiclass',
    'model': 'CTG-Net-1D-Multiclass',
    'steps': CFG['step_numbers'],
    'n_folds': CFG['n_splits'],
    'n_reliable_folds': len(reliable_results),
    'window_length': CFG['segment_length'],
    'n_windows': len(X_all),
    'n_records': int(len(np.unique(groups_all))),
    'n_record_steps': int(len(step_unit_table)),
    'manual_class_weights': manual_class_weights,
    'ctg_net_dropout': CFG['ctg_net_dropout'],
    'mean_macro_f1': float(np.mean([r['macro_f1'] for r in ctg_net_results])),
    'std_macro_f1': float(np.std([r['macro_f1'] for r in ctg_net_results])),
    'mean_balanced_acc': float(np.mean([r['balanced_accuracy'] for r in ctg_net_results])),
    'std_balanced_acc': float(np.std([r['balanced_accuracy'] for r in ctg_net_results])),
    'mean_recall_Normal': float(np.mean([r['recall_Normal'] for r in ctg_net_results])),
    'mean_recall_Mild': float(np.mean([r['recall_Mild'] for r in ctg_net_results])),
    'mean_recall_Severe': float(np.mean([r['recall_Severe'] for r in ctg_net_results])),
    'per_fold': [
        {k: v for k, v in r.items() if k != 'confusion_matrix'}
        for r in ctg_net_results
    ],
}

if reliable_results:
    ctg_net_summary['reliable_mean_macro_f1'] = float(np.mean([r['macro_f1'] for r in reliable_results]))
    ctg_net_summary['reliable_mean_balanced_acc'] = float(np.mean([r['balanced_accuracy'] for r in reliable_results]))

with open(OUTPUT_DIR / "ctg_net_results.json", 'w') as f:
    json.dump(ctg_net_summary, f, indent=2, default=str)

print(f"Mean macro F1:          {ctg_net_summary['mean_macro_f1']:.4f}")
print(f"Mean balanced accuracy: {ctg_net_summary['mean_balanced_acc']:.4f}")
print(f"Mean Mild recall:       {ctg_net_summary['mean_recall_Mild']:.4f}")
print(f"Mean Severe recall:     {ctg_net_summary['mean_recall_Severe']:.4f}")
print(f"\nResults saved to {OUTPUT_DIR / 'ctg_net_results.json'}")

## 12. Results Summary

In [ ]:
def _fmt(vals):
    return f"{np.mean(vals):.4f} +/- {np.std(vals):.4f}"

print("=" * 70)
print("FINAL SUMMARY - CTG-Net 3-Class Classification")
print("=" * 70)
print(f"\nModel: CTG-Net (1D adapted, FHR-only input, {NUM_CLASSES}-class softmax)")
print(f"Steps: {CFG['step_numbers']} | Window: {CFG['segment_length']} samples ({CFG['segment_minutes']} min)")
print(f"Manual class weights: {manual_class_weights}")

print(f"\nPer-fold reliability (min Severe step-units = {CFG['min_severe_records']}):")
print(f"  {'Fold':<8} {'Macro F1':>12} {'Reliable?':>12}  {'Severe step-units':>18}")
for i in range(CFG['n_splits']):
    fold = ctg_net_results[i]
    rel = "Yes" if fold.get('is_reliable', True) else "No"
    sev_n = fold.get('n_severe_val_step_units', '?')
    print(f"  Fold {i + 1:<3} {fold['macro_f1']:>12.4f} {rel:>12}  {sev_n:>18}")

print(f"\nRecord-step metrics across all folds:")
for metric_name in ['accuracy', 'balanced_accuracy', 'macro_f1']:
    vals = [r[metric_name] for r in ctg_net_results]
    print(f"  {metric_name:<22} {_fmt(vals)}")

print("\nPer-class recall:")
for name in CLASS_NAMES:
    vals = [r[f'recall_{name}'] for r in ctg_net_results]
    print(f"  {name:<22} {_fmt(vals)}")

reliable_folds = [r for r in ctg_net_results if r.get('is_reliable', True)]
if 0 < len(reliable_folds) < CFG['n_splits']:
    print(f"\nReliable folds only ({len(reliable_folds)}/{CFG['n_splits']}):")
    for metric_name in ['accuracy', 'balanced_accuracy', 'macro_f1']:
        vals = [r[metric_name] for r in reliable_folds]
        print(f"  {metric_name:<22} {_fmt(vals)}")
elif len(reliable_folds) == CFG['n_splits']:
    print("\nAll folds met the validation reliability threshold.")

print(f"\nResults saved to {OUTPUT_DIR}")